In [3]:
import laspy
import numpy as np
import pandas as pd
import lasio
import matplotlib.pyplot as plt
import plotly.express as px


In [4]:
df = pd.read_excel("Formation Tops.xlsx")
df["Top depth [mMD_RKB]"] = df["Top depth [mMD_RKB]"].astype('float64')

FileNotFoundError: [Errno 2] No such file or directory: 'Formation Tops.xlsx'

In [ ]:
df

In [ ]:
las = lasio.read("Well Log.LAS")

In [ ]:
df_logs = las.df()
df_logs = df_logs.replace(-999.25, np.nan)
df_logs


In [ ]:
# Nulls først (se punkt 3)
df_logs["Vp"] = (1e6 / df_logs["AC"]) * 0.3048      # m/s
df_logs["Vs"] = (1e6 / df_logs["ACS"]) * 0.3048     # m/s

df_logs["AI"] = df_logs["Vp"] * (df_logs["DEN"])  # (m/s)*(kg/m3)
df_logs["Vp/Vs"] = df_logs["Vp"] / df_logs["Vs"]


In [ ]:
df_logs.columns

In [ ]:
plt.figure(figsize=(6,8))
plt.scatter(df_logs["AI"], df_logs["Vp/Vs"], c='blue', s=10, alpha=0.7)
plt.xlabel("Acoustic Impedance (kg/(m²·s))")
plt.ylabel("Vp/Vs ratio")
plt.title("Vp/Vs vs Acoustic Impedance")
plt.grid(True)
plt.show()


In [ ]:
df_logs_with_form = pd.merge_asof(
    df_logs.sort_values("DEPT"),
    df,
    left_on="DEPT",
    right_on="Top depth [mMD_RKB]",
    direction="backward"
)


In [ ]:
df_logs_with_form

In [ ]:
df_clean = df_logs_with_form.rename(columns={"Top depth [mMD_RKB]": "Depth"})


In [ ]:
df_clean

In [ ]:
df_clean['Formation_code'] = pd.Categorical(df_clean['Formation']).codes

# Simple scatter
plt.figure(figsize=(6,8))
plt.scatter(df_clean['AI'], df_clean['Vp/Vs'], c=df_clean['Formation_code'], cmap='tab20', s=20)
plt.gca() # depth usually goes downward
plt.xlabel('AI')      # replace with your X-axis
plt.ylabel('Vp/Vs')
plt.show()


In [ ]:
import plotly.express as px
import pandas as pd

# Check if all data points for each formation have the same depth
depth_counts = df_clean.groupby('Formation')['Depth'].nunique()
print("Number of unique depths per formation:")
print(depth_counts)

# Also show min and max depth per formation
depth_ranges = df_clean.groupby('Formation')['Depth'].agg(['min', 'max', 'count'])


# Reset index to make DEPT a column
df_clean = df_clean.reset_index()

# Add Formation_code to df_clean
df_clean['Formation_code'] = df_clean['Formation'].astype(str)

fig = px.scatter_3d(
    df_clean,
    x='AI',
    z='DEPT',  # use actual log depth for varying depths within formations
    y='Vp/Vs',
    color='Formation_code',   # color by formation
    size_max=5
)

fig.update_layout(scene=dict(zaxis=dict(autorange='reversed')))  # Depth goes downward
fig.show()

In [ ]:
#Ca, egenlagd template basert på ødegaard sine empiriske data.
shale = [(3900, 2.96), (4500, 2.7), (5000, 2.6), (6200, 2.38)]

clean_sand = [(4400, 2.05), (5800, 1.95), (7500, 1.88), (9700, 1.5)]

shale_x, shale_y = zip(*shale)
sand_x, sand_y = zip(*clean_sand)

plt.figure(figsize=(7,5))
plt.scatter(shale_x, shale_y, color='brown', label='Shale', s=80)
plt.scatter(sand_x, sand_y, color='yellow', label='Clean Sand', s=80)
plt.plot(shale_x, shale_y, color='brown', linestyle='-', linewidth=2)
plt.plot(sand_x, sand_y, color='orange', linestyle='-', linewidth=2)
plt.xlabel('AI')
plt.ylabel('Vp/Vs')
plt.title('Vp/Vs vs AI')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
shale = [(3900, 2.96), (4500, 2.7), (5000, 2.6), (6200, 2.38)]
clean_sand = [(4400, 2.05), (5800, 1.95), (7500, 1.88), (9700, 1.5)]

shale_x, shale_y = zip(*shale)
sand_x, sand_y = zip(*clean_sand)

# Scatter from your dataframe
# Assuming df_clean has columns: 'AI', 'Vp/Vs', 'Formation'
df_clean['Formation_code'] = pd.Categorical(df_clean['Formation']).codes

plt.figure(figsize=(8,6))

# Scatter of actual data (colored by formation)
plt.scatter(df_clean['AI'], df_clean['Vp/Vs'], 
            c=df_clean['Formation_code'], cmap='tab20', s=20, alpha=0.7)

# Plot trend lines (shale and sand)
plt.plot(shale_x, shale_y, color='brown', linestyle='-', linewidth=2, label='Shale Trend')
plt.plot(sand_x, sand_y, color='orange', linestyle='-', linewidth=2, label='Clean Sand Trend')

plt.xlabel('AI')
plt.ylabel('Vp/Vs')
plt.title('Vp/Vs vs AI with Trend Lines')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
shale = [(3900, 2.96), (4500, 2.7), (5000, 2.6), (6200, 2.38)]
clean_sand = [(4400, 2.05), (5800, 1.95), (7500, 1.88), (9700, 1.5)]

shale_x, shale_y = zip(*shale)
sand_x, sand_y = zip(*clean_sand)

# Scatter from your dataframe
# Assuming df_clean has columns: 'AI', 'Vp/Vs', 'Formation'
df_clean['GR_color'] = pd.Categorical(df_clean['GR']).codes

plt.figure(figsize=(8,6))

# Scatter of actual data (colored by formation)
plt.scatter(df_clean['AI'], df_clean['Vp/Vs'], 
            c=df_clean['GR_color'], cmap='tab20', s=20, alpha=0.7)

# Plot trend lines (shale and sand)
plt.plot(shale_x, shale_y, color='brown', linestyle='-', linewidth=2, label='Shale Trend')
plt.plot(sand_x, sand_y, color='orange', linestyle='-', linewidth=2, label='Clean Sand Trend')

plt.xlabel('AI')
plt.ylabel('Vp/Vs')
plt.title('Vp/Vs vs AI with Trend Lines')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(8,6))

# Scatter of actual data (colored by GR values now, using a continuous colormap)
plt.scatter(df_clean['AI'], df_clean['Vp/Vs'], 
            c=df_clean['GR'], cmap='viridis', s=20, alpha=0.7)  # Changed 'GR_color' to 'GR' and 'tab20' to 'viridis' for continuous coloring

# Plot trend lines (shale and sand)
plt.plot(shale_x, shale_y, color='brown', linestyle='-', linewidth=2, label='Shale Trend')
plt.plot(sand_x, sand_y, color='orange', linestyle='-', linewidth=2, label='Clean Sand Trend')

plt.xlabel('AI')
plt.ylabel('Vp/Vs')
plt.title('Vp/Vs vs AI with Trend Lines')
plt.legend()
plt.grid(True)

# Add color bar at the bottom
plt.colorbar(orientation='horizontal', label='GR Values')

plt.show()
